# Figure 3 — system prompts

The exact `system_prompt` recorded for each arm on one case (copied verbatim from the
`results/full/*.jsonl` rows), stacked as three panels. A real `difflib` diff against the
naked arm (whose prompt is the shared medicinal-chemistry block alone) highlights what each
arm adds: the access mechanics and its tool guidance. Saved to
`notebooks/system-prompts-figure.png`.

In [1]:
%matplotlib inline
import json, textwrap, difflib
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / 'benchmarks').is_dir():
        break
    ROOT = ROOT.parent

TARGET, TRACK, CASE = 'egfr', '2d', 'grow-cyclopropyl'
ARMS = ['naked', 'generalist', 'chemistree']

def system_prompt(arm):
    """The exact recorded system_prompt for this arm on the chosen case."""
    f = ROOT / f'benchmarks/results/full/{TARGET}/{TRACK}_{arm}.jsonl'
    for line in open(f):
        r = json.loads(line)
        if r['id'] == CASE:
            return r['system_prompt']
    raise SystemExit(f'{CASE} not found in {f}')

prompts = {a: system_prompt(a) for a in ARMS}
for a in ARMS:
    print(a, len(prompts[a]), 'chars')

naked 3801 chars
generalist 4784 chars
chemistree 10930 chars


In [2]:
import re

MAXCHARS = 92   # re-flow width: joins the source's hard-wraps and re-wraps cleanly so lines
                # fill the column (no orphan single-word lines). No words are added or removed.

def reflow(text):
    """Re-wrap prose to MAXCHARS while preserving headers, blank lines, list items, and
    fenced code blocks. Only whitespace/line breaks change; every word is kept."""
    out, para = [], []
    lst = [None]        # (marker, body) for the current list item, or None
    in_code = [False]

    def flush_para():
        if para:
            joined = " ".join(s.strip() for s in para)
            out.extend(textwrap.wrap(joined, MAXCHARS, break_long_words=True,
                                     break_on_hyphens=False) or [""])
            para.clear()

    def flush_list():
        if lst[0] is not None:
            marker, body = lst[0]
            w = textwrap.wrap(body, MAXCHARS - len(marker), break_long_words=True,
                              break_on_hyphens=False) or [""]
            out.append(marker + w[0])
            out.extend(" " * len(marker) + piece for piece in w[1:])
            lst[0] = None

    for ln in text.split("\n"):
        s = ln.strip()
        if s.startswith("```"):
            flush_para(); flush_list(); out.append(ln); in_code[0] = not in_code[0]; continue
        if in_code[0]:
            out.append(ln); continue
        if s == "":
            flush_para(); flush_list(); out.append(""); continue
        if s.startswith("#"):
            flush_para(); flush_list(); out.append(ln); continue
        m = re.match(r"^([-*]\s+|\d+\.\s+)(.*)", ln)
        if m:
            flush_para(); flush_list()
            lst[0] = (m.group(1), m.group(2).strip())
        elif lst[0] is not None:
            lst[0] = (lst[0][0], (lst[0][1] + " " + s).strip())   # list-item continuation
        else:
            para.append(ln)
    flush_para(); flush_list()
    return out

# Re-flow each arm, then run the real diff on the re-flowed lines. The shared medchem base
# re-flows identically in every arm, so the diff still isolates each arm's added lines.
reflowed = {a: reflow(prompts[a]) for a in ARMS}
base = reflowed["naked"]

def unique_flags(arm):
    lines = reflowed[arm]
    flags = [False] * len(lines)
    for tag, _i1, _i2, j1, j2 in difflib.SequenceMatcher(None, base, lines).get_opcodes():
        if tag in ("insert", "replace"):
            for j in range(j1, j2):
                flags[j] = True
    return lines, flags

diffed = {a: unique_flags(a) for a in ARMS}
for a in ARMS:
    print(a, "unique lines:", sum(diffed[a][1]))

naked unique lines: 0
generalist unique lines: 24
chemistree unique lines: 100


In [3]:
import math
# ---- layout (inches): two columns, font 6, lines pre-flowed to the column width ----
W, FONT, LINE_H = 10.6, 6.0, 0.125
NCOL, COLGAP = 2, 0.5
TITLE_H, PADX, PADY, VGAP, MARG = 0.44, 0.18, 0.16, 0.5, 0.35
ACCENT = {"naked": "#8a8a8a", "generalist": "#1a7f37", "chemistree": "#1f6feb"}
HILITE = {"naked": "#eeeeee", "generalist": "#dafbe1", "chemistree": "#dbeafe"}
CW = (W - 2 * PADX - (NCOL - 1) * COLGAP) / NCOL

panels = {a: list(zip(diffed[a][0], diffed[a][1])) for a in ARMS}
rows_per = {a: math.ceil(len(panels[a]) / NCOL) for a in ARMS}
panel_h = {a: TITLE_H + PADY + rows_per[a] * LINE_H + PADY for a in ARMS}
figW = 2 * MARG + W
figH = 2 * MARG + sum(panel_h[a] for a in ARMS) + VGAP * (len(ARMS) - 1)

fig = plt.figure(figsize=(figW, figH), dpi=150)
fig.patch.set_facecolor("white")
fx, fy = 1.0 / figW, 1.0 / figH

top = MARG
for a in ARMS:
    h, edge, rper = panel_h[a], ACCENT[a], rows_per[a]
    fig.add_artist(FancyBboxPatch((MARG * fx, (figH - top - h) * fy), W * fx, h * fy,
                   boxstyle="round,pad=0,rounding_size=0.004", linewidth=2.0,
                   edgecolor=edge, facecolor="white", transform=fig.transFigure, zorder=1))
    n_uni = sum(uq for _, uq in panels[a])
    title = f"{a}   (system_prompt, {len(prompts[a])} chars" + \
            (f"; {n_uni} lines unique to this arm)" if n_uni else "; = shared base, nothing unique)")
    fig.text((MARG + PADX) * fx, (figH - (top + TITLE_H * 0.6)) * fy, title,
             ha="left", va="center", fontsize=11, fontweight="bold", color=edge, zorder=3)
    ty = top + TITLE_H + PADY
    for idx, (s, uq) in enumerate(panels[a]):
        col, row = divmod(idx, rper)
        cx = MARG + PADX + col * (CW + COLGAP)
        y = ty + row * LINE_H
        if uq:
            fig.add_artist(Rectangle((cx * fx, (figH - (y + LINE_H)) * fy), CW * fx, LINE_H * fy,
                           facecolor=HILITE[a], edgecolor="none", transform=fig.transFigure, zorder=2))
        fig.text((cx + 0.03) * fx, (figH - (y + LINE_H * 0.5)) * fy, s, ha="left", va="center",
                 family="monospace", fontsize=FONT, color="#111111", zorder=3)
    top += h + VGAP

OUT = ROOT / "notebooks"
fig.savefig(OUT / "system-prompts-figure.png", dpi=150, facecolor="white")
print("saved", OUT / "system-prompts-figure.png", "|", round(figW, 1), "x", round(figH, 1), "in")
plt.show()

saved /home/ben/packages/chemistree/notebooks/system-prompts-figure.png | 11.3 x 23.7 in


<Figure size 1695x3559.5 with 0 Axes>